In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [2]:
DATA_PATH = os.path.join("../../data/", "santander-customer-satisfaction")

train_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))

print(f"Train 데이터 크기: {train_df.shape}")

train_df.head()

Train 데이터 크기: (76020, 371)


,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.170000,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.030000,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.770000,0
3,8,2,37,0.0,195.0,195.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,64007.970000,0
4,10,2,39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117310.979016,0


In [3]:
# 정답 라벨 분리 (맨 마지막 컬럼 TARGET)
y_labels = train_df.iloc[:, -1].copy()

# ID와 TARGET을 제외한 순수 피처 세트 분리
X_features = train_df.drop(columns=['ID', 'TARGET']).copy()

print(f"X_features Shape: {X_features.shape}")
print(f"y_labels Shape: {y_labels.shape}")

X_features Shape: (76020, 369)
y_labels Shape: (76020,)


In [4]:
# var3 결측성 이상치(-999999)를 최빈값(2)으로 치환
X_features['var3'] = X_features['var3'].replace(-999999, 2)

print("var3 이상치 치환 완료 (-999999 잔여 개수):", (X_features['var3'] == -999999).sum())

var3 이상치 치환 완료 (-999999 잔여 개수): 0


In [5]:
# var38 자산 왜도 완화를 위한 로그 변환
X_features['var38'] = np.log1p(X_features['var38'])

print("var38 로그 변환 완료")

var38 로그 변환 완료


In [6]:
# 고객별 0의 개수 카운트
X_features['n0'] = (X_features == 0).sum(axis=1)

# 고객별 거래/잔액의 표준편차
X_features['row_std'] = X_features.std(axis=1)

print("행 통계량 파생 변수(n0, row_std) 생성 완료")

행 통계량 파생 변수(n0, row_std) 생성 완료


In [7]:
# 1. 분산 0인 상수 컬럼 제거 <- 제거 한 것과 안 한 것 둘 다 결과 같음
zero_var_cols = [col for col in X_features.columns if X_features[col].nunique() == 1]
X_features.drop(columns=zero_var_cols, inplace=True)
print(f"1) 제거된 상수 컬럼 수: {len(zero_var_cols)}")

# 2. 중복 컬럼 초고속 제거
dup_cols = X_features.T.duplicated()
dup_col_names = X_features.columns[dup_cols].tolist()
X_features.drop(columns=dup_col_names, inplace=True)
print(f"2) 제거된 중복 컬럼 수: {len(dup_col_names)}")

# 3. var6 유사 피처 및 다중공선성 delta 컬럼 제거
manual_remove = [c for c in X_features.columns if 'var6' in c] + [
    'delta_imp_reemb_var13_1y3', 'delta_imp_reemb_var17_1y3', 
    'delta_imp_trasp_var17_in_1y3', 'delta_imp_trasp_var33_in_1y3'
]
manual_remove = [c for c in manual_remove if c in X_features.columns]
X_features.drop(columns=manual_remove, inplace=True)
print(f"3) 추가 제거된 유사/노이즈 컬럼 수: {len(manual_remove)}")



1) 제거된 상수 컬럼 수: 34
2) 제거된 중복 컬럼 수: 29
3) 추가 제거된 유사/노이즈 컬럼 수: 9


In [ ]:
# 1차 분할: 학습 세트(80%)와 최종 테스트 세트(20%)로 분리
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_labels,
    test_size=0.2, 
    stratify=y_labels, 
    random_state=0
)

# 2차 분할: 학습 세트를 다시 훈련용(70%)과 조기 중단 감시용(30%)으로 분리
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.3, 
    stratify=y_train, 
    random_state=0
)

print(f"훈련 세트(X_tr) Shape: {X_tr.shape}")
print(f"검증 세트(X_val) Shape: {X_val.shape}")
print(f"테스트 세트(X_test) Shape: {X_test.shape}")

훈련 세트(X_tr) Shape: (42571, 299)
검증 세트(X_val) Shape: (18245, 299)
테스트 세트(X_test) Shape: (15204, 299)


In [9]:
# (1) 훈련 데이터(X_tr)로만 베이스 모델 학습하여 피처 중요도 측정
base_xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=156,
    n_jobs=-1
)
base_xgb.fit(X_tr, y_tr)

# (2) 중요도가 0인 피처 확인 후 3개 세트(X_tr, X_val, X_test)에서 동일하게 제거
feat_imp = pd.Series(base_xgb.feature_importances_, index=X_tr.columns)
zero_imp_cols = feat_imp[feat_imp == 0].index.tolist()

X_tr = X_tr.drop(columns=zero_imp_cols)
X_val = X_val.drop(columns=zero_imp_cols)
X_test = X_test.drop(columns=zero_imp_cols)
print(f"제거된 중요도 0인 컬럼 수: {len(zero_imp_cols)}")

# (3) X_tr 기준으로 StandardScaler 학습(fit) 후 변환(transform)
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# (4) X_tr 기준으로 PCA 학습(fit) 후 주성분 5개 변환(transform)
pca = PCA(n_components=5, random_state=156)
pca_tr = pca.fit_transform(X_tr_scaled)
pca_val = pca.transform(X_val_scaled)
pca_test = pca.transform(X_test_scaled)

# (5) 각 세트에 pca1 ~ pca5 컬럼 추가
for i in range(5):
    X_tr[f'pca{i+1}'] = pca_tr[:, i]
    X_val[f'pca{i+1}'] = pca_val[:, i]
    X_test[f'pca{i+1}'] = pca_test[:, i]

print(f"훈련 세트(X_tr) 최종 Shape: {X_tr.shape}")
print(f"검증 세트(X_val) 최종 Shape: {X_val.shape}")
print(f"테스트 세트(X_test) 최종 Shape: {X_test.shape}")

제거된 중요도 0인 컬럼 수: 188
훈련 세트(X_tr) 최종 Shape: (42571, 116)
검증 세트(X_val) 최종 Shape: (18245, 116)
테스트 세트(X_test) 최종 Shape: (15204, 116)


In [10]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# 1. GBM 모델 정의
# - n_iter_no_change: 조기 중단(Early Stopping)을 위한 라운드 수
# - validation_fraction: 내부 검증에 쓸 비율 (0.2)
gbm_clf = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    validation_fraction=0.2,
    n_iter_no_change=20,
    random_state=156
)

# 2. 모델 학습 (X_tr, y_tr 입력)
# 트리 모델이므로 스케일링되지 않은 X_tr 데이터를 그대로 사용합니다.
gbm_clf.fit(X_tr, y_tr)

# 3. 검증 및 최종 테스트 세트 점수 산출
val_preds = gbm_clf.predict_proba(X_val)[:, 1]
test_preds = gbm_clf.predict_proba(X_test)[:, 1]

gbm_val_auc = roc_auc_score(y_val, val_preds)
gbm_test_auc = roc_auc_score(y_test, test_preds)

print("=" * 40)
print(f"GBM 검증 세트 ROC-AUC: {gbm_val_auc:.4f}")
print(f"GBM 최종 테스트 ROC-AUC: {gbm_test_auc:.4f}")
print("=" * 40)

GBM 검증 세트 ROC-AUC: 0.8420
GBM 최종 테스트 ROC-AUC: 0.8188
